# Two Adaption Implementation

In this section we implemented two methods, `LoRa `and `LoRa+` in
order to adapt the transformer on the specific task of sentiment analysis and compare the training time, inference time and the performance with our foundation model `bert-base-cased`. \

In [ ]:
!pip install torch datasets transformers peft
!pip install accelerate>=0.26.0
!pip install evaluate scikit-learn

In [ ]:
import time
import os
os.environ["TORCH_USE_CUDA_DSA"] = "1"
os.environ["WANDB_DISABLED"] = "true"
import torch
import transformers
from torch import nn
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
import numpy as np
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score


# Define compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions)
    recall = recall_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall}



# Load dataset
dataset = load_dataset("stanfordnlp/imdb")

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

dataset = dataset.map(tokenize_function, batched=True)

def get_model(peft_config=None):
    model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
    if peft_config:
        model = get_peft_model(model, peft_config)
    return model

# Define PEFT configurations
lora_config = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, lora_dropout=0.1)
lora_plus_config = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=32, lora_dropout=0.1, bias="none", target_modules=["query", "value"])

def train_and_evaluate(model, dataset, approach):
    training_args = TrainingArguments(
        output_dir=f"./results_{approach}",
        evaluation_strategy="steps",
        save_strategy="steps",
        eval_steps=128,
        save_steps=128,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_dir=f"./logs_{approach}",
        logging_steps=128,
        learning_rate=5e-5,
        gradient_checkpointing=False,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        compute_metrics=compute_metrics,
        )
    start_time = time.time()
    trainer.train()
    train_time = time.time() - start_time
    eval_results = trainer.evaluate()
    return train_time, eval_results


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
# Dictionary to store metrics
results = {}

In [ ]:
# Train and Evaluate Full Fine-tuning version
model_ft = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)
ft_time, ft_results = train_and_evaluate(model_ft, dataset, "finetuning")
results["finetuning"] = {"time": ft_time, "metrics": ft_results}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall
128,0.382700,0.289390,0.887680,0.852539,0.937520
256,0.280800,0.243616,0.907880,0.920218,0.893200
384,0.263600,0.220650,0.913240,0.907728,0.920000
512,0.231200,0.198272,0.924760,0.937464,0.910240
640,0.225700,0.200459,0.920680,0.937443,0.901520
768,0.212100,0.187753,0.929800,0.920482,0.940880
896,0.119800,0.237860,0.932240,0.922969,0.943200
1024,0.126100,0.234955,0.929440,0.948604,0.908080
1152,0.108100,0.211371,0.934400,0.940604,0.927360
1280,0.104500,0.219612,0.935040,0.937138,0.932640


In [ ]:
# Train and evaluate LoRA
model_lora = get_model(lora_config)
lora_time, lora_results = train_and_evaluate(model_lora, dataset, "lora")
results["lora"] = {"time": lora_time, "metrics": lora_results}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall
128,0.669100,0.596311,0.699640,0.703964,0.689040
256,0.468200,0.332759,0.858960,0.880965,0.830080
384,0.327900,0.296403,0.877320,0.895779,0.854000
512,0.301100,0.295427,0.878320,0.849055,0.920240
640,0.298900,0.274525,0.886240,0.870473,0.907520
768,0.290400,0.267220,0.890880,0.917393,0.859120
896,0.276000,0.264682,0.891320,0.874283,0.914080
1024,0.268800,0.255704,0.896120,0.898768,0.892800
1152,0.260400,0.256999,0.895840,0.884819,0.910160
1280,0.270000,0.251806,0.897320,0.896654,0.898160


In [ ]:
# Justification for LoRA+
print("""
Justification for LoRA+

LoRA+ improves upon standard LoRA by setting different learning rates for matrices A and B.
Instead of using the same learning rate for both, LoRA+ assigns:

- `lr_A = 5e-5` (same as standard LoRA)
- `lr_B = 24 × lr_A = 1.2e-3` (higher learning rate for B)

This adjustment ensures better feature learning and faster convergence.
Studies show that LoRA+ can improve accuracy by 1-2% while reducing training time.

Applying modified learning rates now...
""")

# Load LoRA+ model
model_lora_plus = get_model(lora_plus_config)

# Create Trainer as usual
trainer_lora_plus = Trainer(
    model=model_lora_plus,
    args=TrainingArguments(
        output_dir="./results_lora_plus",
        evaluation_strategy="steps",
        save_strategy="steps",
        eval_steps=128,
        save_steps=128,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_dir="./logs_lora_plus",
        logging_steps=128,
        learning_rate=5e-5,  # Base learning rate (for A)
        gradient_checkpointing=False,
        report_to="none",
    ),
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)

# Manually Adjust Learning Rate for B Matrices
lambda_ratio = 24  # Recommended LoRA+ ratio
lr_b = lambda_ratio * 5e-5  # Higher LR for B

for name, param in model_lora_plus.named_parameters():
    if "lora_B" in name:
        param.lr = lr_b  # Set higher LR for B

# Train and evaluate LoRA+
start_time = time.time()
trainer_lora_plus.train()
lora_plus_time = time.time() - start_time
lora_plus_results = trainer_lora_plus.evaluate()
results["lora_plus"] = {"time": lora_plus_time, "metrics": lora_plus_results}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Justification for LoRA+ 

LoRA+ improves upon standard LoRA by setting different learning rates for matrices A and B.
Instead of using the same learning rate for both, LoRA+ assigns:

- `lr_A = 5e-5` (same as standard LoRA)
- `lr_B = 24 × lr_A = 1.2e-3` (higher learning rate for B)

This adjustment ensures better feature learning and faster convergence. 
Studies show that LoRA+ can improve accuracy by 1-2% while reducing training time.

Applying modified learning rates now...



/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall
128,0.646700,0.505867,0.765240,0.755569,0.784160
256,0.373400,0.299290,0.878240,0.886654,0.867360
384,0.310400,0.277822,0.887000,0.886907,0.887120
512,0.285700,0.272208,0.888320,0.866949,0.917440
640,0.287300,0.258487,0.893720,0.877213,0.915600
768,0.279300,0.253140,0.898680,0.918958,0.874480
896,0.264400,0.255952,0.894240,0.866994,0.931360
1024,0.255000,0.245088,0.900120,0.885531,0.919040
1152,0.248600,0.246536,0.899400,0.878535,0.926960
1280,0.258100,0.238601,0.902600,0.894304,0.913120


In [ ]:
# Print comparison
print("\n--- Evaluation Summary ---")
for method, data in results.items():
    print(f"\nMethod: {method}")
    print(f"  Training Time: {data['time']:.2f} seconds")
    for metric, value in data['metrics'].items():
        print(f"  {metric}: {value:.4f}")


--- Evaluation Summary ---

Method: finetuning
  Training Time: 2940.13 seconds
  eval_loss: 0.2199
  eval_accuracy: 0.9360
  eval_precision: 0.9373
  eval_recall: 0.9346
  eval_runtime: 162.9857
  eval_samples_per_second: 153.3880
  eval_steps_per_second: 4.7980
  epoch: 2.0000

Method: lora
  Training Time: 2738.39 seconds
  eval_loss: 0.2520
  eval_accuracy: 0.8974
  eval_precision: 0.8900
  eval_recall: 0.9070
  eval_runtime: 168.8208
  eval_samples_per_second: 148.0860
  eval_steps_per_second: 4.6320
  epoch: 2.0000

Method: lora_plus
  Training Time: 2739.33 seconds
  eval_loss: 0.2396
  eval_accuracy: 0.9027
  eval_precision: 0.8889
  eval_recall: 0.9205
  eval_runtime: 168.8700
  eval_samples_per_second: 148.0430
  eval_steps_per_second: 4.6310
  epoch: 2.0000
